In [ ]:
import os
import random
from dataclasses import dataclass
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

@dataclass
class Config:
    data_dir: str = r"realwaste-main/RealWaste" 
    image_size: int = 300 
    batch_size: int = 16    
    lr: float = 3e-4
    epochs: int = 20        
    seed: int = 42
    num_workers: int = 2
    num_classes: int = 9 
    out_weights: str = "weights_efficientnet_b0_bias_corrected.pt"

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_transforms(image_size: int):
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
        
        transforms.RandomGrayscale(p=0.4), 
        
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        
        transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
        transforms.RandomAutocontrast(p=0.3),
        
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    
    val_tfms = transforms.Compose([
        transforms.Resize(image_size + 32),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    return train_tfms, val_tfms

def build_model(num_classes: int):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(y.cpu().numpy())
    
    y_p = np.concatenate(all_preds)
    y_t = np.concatenate(all_targets)
    return accuracy_score(y_t, y_p), f1_score(y_t, y_p, average="macro")

def main():
    cfg = Config()
    set_seed(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_tfms, val_tfms = make_transforms(cfg.image_size)

    full_dataset = datasets.ImageFolder(cfg.data_dir, transform=train_tfms)
    print(f"Detected Classes: {full_dataset.class_to_idx}")

    indices = list(range(len(full_dataset)))
    train_idx, val_idx = train_test_split(
        indices, test_size=0.2, random_state=cfg.seed, stratify=full_dataset.targets
    )

    train_ds = Subset(full_dataset, train_idx)
    val_ds = Subset(datasets.ImageFolder(cfg.data_dir, transform=val_tfms), val_idx)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    targets = [full_dataset.targets[i] for i in train_idx]
    counts = np.bincount(targets)
    
    weights = 1.0 / counts
    
    if len(weights) > 4:
        print(f"Applying penalty to class: {full_dataset.classes[4]}")
        weights[4] = weights[4] * 0.8 
        
    class_weights = torch.tensor(weights / weights.sum() * cfg.num_classes, dtype=torch.float32, device=device)

    model = build_model(cfg.num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    
    best_f1 = 0.0
    for epoch in range(1, cfg.epochs + 1):
        for param in model.features.parameters():
            param.requires_grad = (epoch > 3)

        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        acc, f1 = evaluate(model, val_loader, device)
        print(f"Epoch {epoch} | Loss: {train_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), cfg.out_weights)
            print(f"--> Saved New Best Weights (F1: {f1:.4f})")

if __name__ == "__main__":
    main()

Detected Classes: {'Cardboard': 0, 'Food Organics': 1, 'Glass': 2, 'Metal': 3, 'Miscellaneous Trash': 4, 'Paper': 5, 'Plastic': 6, 'Textile Trash': 7, 'Vegetation': 8}
Applying penalty to class: Miscellaneous Trash


Epoch 1: 100%|██████████| 252/252 [02:06<00:00,  1.99it/s]


Epoch 1 | Loss: 1.9205 | Acc: 0.6514 | F1: 0.6504
--> Saved New Best Weights (F1: 0.6504)


Epoch 2: 100%|██████████| 252/252 [01:48<00:00,  2.33it/s]


Epoch 2 | Loss: 1.5789 | Acc: 0.6753 | F1: 0.6765
--> Saved New Best Weights (F1: 0.6765)


...

Epoch 18 | Loss: 0.7982 | Acc: 0.9384 | F1: 0.9422


Epoch 19: 100%|██████████| 252/252 [04:08<00:00,  1.01it/s]


Epoch 19 | Loss: 0.7963 | Acc: 0.9414 | F1: 0.9444


Epoch 20: 100%|██████████| 252/252 [04:14<00:00,  1.01s/it]


Epoch 20 | Loss: 0.7991 | Acc: 0.9265 | F1: 0.9302


In [ ]:
from icrawler.builtin import BingImageCrawler, BaiduImageCrawler
from icrawler import ImageDownloader
import os

class PrefixNameDownloader(ImageDownloader):
    def get_filename(self, task, default_ext):
        prefix = self.storage.prefix if hasattr(self.storage, 'prefix') else 'image'
        filename = super().get_filename(task, default_ext)
        return f"{prefix}_{filename}"

def get_more_wrappers(base_dir, target_total=200):
    target_path = os.path.join(base_dir, 'MiscTrash')
    
    current_count = len([f for f in os.listdir(target_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
    needed = target_total - current_count
    
    if needed <= 0:
        print("You already have enough images!")
        return

    sub_types = {
        'Foil_Metallic': [
            'crinkled foil chip bag waste', 
            'metallic candy wrapper on ground', 
            'inside of potato chip bag texture',
            'aluminum foil packaging trash'
        ],
        'Transparent_Film': [
            'clear plastic sandwich bag waste', 
            'discarded transparent bubble wrap', 
            'crumpled cling film trash', 
            'clear plastic shrink wrap waste'
        ],
        'High_Saturated': [
            'colorful granola bar wrapper', 
            'skittles wrapper trash', 
            'colorful plastic snack packaging'
        ],
        'Matte_White': [
            'white plastic mailing bag trash', 
            'matte white grocery bag crumpled', 
            'white poly mailer waste', 
        ]
    }

    print(f"Current count: {current_count}. Aiming for {needed} more...")

    for label, queries in sub_types.items():
        crawler_class = BaiduImageCrawler if 'bulk_trash' in label else BingImageCrawler
        
        crawler = crawler_class(
            downloader_cls=PrefixNameDownloader,
            storage={'root_dir': target_path}
        )
        crawler.storage.prefix = label
        
        for q in queries:
            crawler.crawl(keyword=q, max_num=needed // 2)

get_more_wrappers('realwaste-main/RealWaste', target_total=200)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# 1. Setup Environment
device = "cuda" if torch.cuda.is_available() else "cpu"
image_size = 300 # Matches your training config
weights_path = "weights_efficientnet_b0_final_9cls.pt"
data_dir = "realwaste-main/RealWaste"

# 2. Define Val Transforms (NO random warping/flipping here!)
val_tfms = transforms.Compose([
    transforms.Resize(image_size + 32),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# 3. Load Dataset and Recreate Split
# It's vital we use the same seed (42) as the training script to get the same Val Set
full_dataset = datasets.ImageFolder(data_dir, transform=val_tfms)
class_names = full_dataset.classes # Automatically gets alphabetical names

indices = list(range(len(full_dataset)))
_, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=full_dataset.targets
)
val_ds = Subset(full_dataset, val_idx)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

# 4. Load the Model
from torchvision.models import efficientnet_b0
model = efficientnet_b0()
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 9)
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)
model.eval()

# 5. Run Inference
print("Analyzing validation set...")
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 6. Plotting the Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.title('9-Class Waste Classification Matrix\n(Optimized for Wrappers)', fontsize=14)
plt.show()

# 7. Print Detailed Stats
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))